# Earth4All: Too Little Too Late, and the Giant Leap

The Earth4All model asks what the rest of this century looks like under two
courses of action. In *Too Little Too Late* the world keeps doing what it has
been doing. In the *Giant Leap* it carries out five turnarounds — on poverty,
inequality, empowerment, food and energy — and the model differs between the two
only by the parameters those turnarounds set.

This notebook runs both, reads the difference off the figures and the numbers,
then builds a third scenario to see what one turnaround does on its own.

It needs the package and its plotting extras:

```sh
pip install -e ".[dev]"
```

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from earth4all import run_gl_solution, run_tltl_solution

start = time.perf_counter()
tltl = run_tltl_solution()
gl = run_gl_solution()
print(f"both scenarios solved in {time.perf_counter() - start:.1f} s")
tltl

A solution holds every variable of the model — 638 of them — at each of the 7681
time points between 1980 and 2100. Index it with `"sector.name"`, or with a bare
name when it is unambiguous.

In [ ]:
tltl["pop.POP"][-1], tltl["cli.OW"][-1]  # population in millions, warming in degrees

## The two base runs

Six variables carry the story: how many people there are, how much each of them
produces, how well they live, how much they resent each other, how unequally the
output is shared, and how much hotter it has become. Each is drawn against its
own axis.

In [ ]:
from earth4all import fig_baserun_tltl

fig_baserun_tltl();

In [ ]:
from earth4all import fig_baserun_gl

fig_baserun_gl();

The two are easier to read one variable at a time, the scenarios on top of each
other.

In [ ]:
from earth4all import fig_baserun_comparison

fig_baserun_comparison();

## The numbers behind the figures

In [ ]:
HEADLINE = {
    "Population (Mp)": "pop.POP",
    "GDP per person (k$/p/y)": "pop.GDPP",
    "Average wellbeing": "wel.AWBI",
    "Social tension": "wel.STE",
    "Inequality": "dem.INEQ",
    "Observed warming (°C)": "cli.OW",
}


def values_in(solution, year):
    """The headline variables at the start of `year`."""
    index = int((year - 1980) * 64)
    return {label: solution[name][index] for label, name in HEADLINE.items()}


pd.DataFrame({
    ("Too Little Too Late", 2050): values_in(tltl, 2050),
    ("Too Little Too Late", 2100): values_in(tltl, 2100),
    ("Giant Leap", 2050): values_in(gl, 2050),
    ("Giant Leap", 2100): values_in(gl, 2100),
}).round(2)

Population peaks in both scenarios and then falls, but it peaks earlier and lower
in the Giant Leap, where wellbeing rises instead of drifting down.

In [ ]:
for name, solution in [("Too Little Too Late", tltl), ("Giant Leap", gl)]:
    population = solution["pop.POP"]
    peak = population.argmax()
    warming = solution["cli.OW"]
    print(f"{name}:")
    print(f"    population peaks at {population[peak]:,.0f} Mp in {solution.t[peak]:.0f},"
          f" and is {population[-1]:,.0f} Mp in 2100")
    print(f"    warming reaches {warming.max():.2f} °C, and is {warming[-1]:.2f} °C in 2100")

## A scenario of your own

`run_e4a` takes the parameters of every sector, so a scenario is a copy of one
sector's parameters with a few of them changed. That is all `run_gl` is.

Here only the climate turnaround is switched on — direct air capture, and faster
declines in the nitrous oxide per kilogram of fertiliser and the methane per
kilogram of crop — while everything else stays on the *Too Little Too Late* path.

In [ ]:
from earth4all import climate, run_e4a_solution

parameters = climate.getparameters()
parameters["DACCO22100"] = 8       # direct air capture of CO2 in 2100, GtCO2/y
parameters["ERDN2OKF2022"] = 0.01  # extra rate of decline in N2O per kg fertiliser
parameters["ERDCH4KC2022"] = 0.01  # extra rate of decline in CH4 per kg crop
climate_only = run_e4a_solution(cli_pars=parameters)

runs = [
    ("Too Little Too Late", tltl),
    ("climate policies only", climate_only),
    ("Giant Leap", gl),
]

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, solution in runs:
    axes[0].plot(solution.t, solution["cli.OW"], label=label)
    axes[1].plot(solution.t, solution["wel.AWBI"], label=label)
axes[0].set_title("Observed warming °C")
axes[1].set_title("Average wellbeing")
for panel in axes:
    panel.set_xlim(1980, 2100)
    panel.set_xlabel("Year")
axes[0].legend()
figure.tight_layout();

The climate parameters alone bend warming down by less than a tenth of a degree
by 2100, and leave wellbeing far below where the Giant Leap takes it. In this
model the turnarounds work as a set: the ones that raise wellbeing are also the
ones that slow population growth, and it is the smaller, better-off population
that does most of the work on emissions.

## One sector on its own

A sector can be run by itself, with the variables the other eleven would have
given it read from tabulated runs of the whole model. It is the quickest way to
see what a sector does before changing it.

In [ ]:
from earth4all import climate

climate.fig_cli();

## Agreement with Vensim

The Julia package this one is ported from keeps the trajectories Vensim exports
for both scenarios. `all_mre` compares a run with them variable by variable, over
every time point, matching each variable to its reference series by the name it
carries in Vensim.

Most sectors agree to within a few percent. Demand, energy and the labour market
report much larger relative errors on a handful of rates that cross zero, where
the six digits Vensim exports are not enough to compare a difference of two
nearly equal numbers.

In [ ]:
from pathlib import Path

from earth4all import run_tltl
from earth4all.system import solve
from earth4all.validation import all_mre

vensim_output = Path("../../Earth4All.jl/VensimOutput")

if vensim_output.is_dir():
    model = run_tltl()
    solution = solve(model, (1980, 2100), dt=0.015625)
    _, per_sector = all_mre("TLTL", solution, model.systems, vensim_output)
    display(pd.Series(per_sector, name="largest relative error").sort_values().round(4))
else:
    print(f"{vensim_output} not found:"
          " clone Earth4All.jl next to this repository to run this comparison")

## Where to go next

- `earth4all/<sector>/subsystems.py` holds the equations of a sector, one Vensim
  equation per line.
- `earth4all/<sector>/parameters.py` holds what you can change, with the meaning
  of each parameter next to it.
- `solution.dataframe()` gives the whole run as a pandas table, if you would
  rather take it elsewhere.